## Output ID Consistency Check
Check if the output US_IDs in the generated "dependencies" list (last decision) are consistent with the US_IDs in the stage 1 as well as stage 2, which are inherent correctly from the input data.

In [2]:
from pathlib import Path
import json
import re
import pandas as pd

In [5]:
PREDICTION_DIR = Path("pipeline_outputs")

RESULT_FILES = [
    file_path
    for pattern in ("*annotated_gt_results.json", "*annotated_llm_results.json")
    for file_path in PREDICTION_DIR.glob(pattern)
]

In [9]:
if not RESULT_FILES:
    print(f"No files ending in 'annotated_gt_results.json' or 'annotated_llm_results.json' found in {PREDICTION_DIR}")
else:
    for file_path in RESULT_FILES:
        print(file_path)

pipeline_outputs\llama3.1_8b_annotated_gt_results.json
pipeline_outputs\qwen3_8b_annotated_gt_results.json
pipeline_outputs\qwen3_8b_annotated_llm_results.json


In [7]:
def validate_dependency_ids(results, model_name):
    """
    Validate that final_llm dependency IDs only use the IDs provided by
    stage 1 and stage 2 for the same story pair.
    """
    issues = []

    for item_index, item in enumerate(results):
        stage_1 = item["step_1_match"]

        stage_1_ids = {
            stage_1["story_a_id"],
            stage_1["story_b_id"],
        }

        # The LLM may only use IDs provided by Stage 1.
        dependencies = item["final_llm"]["dependencies"]

        for dependency_index, dependency in enumerate(dependencies):
            output_ids = {
                dependency.get("prerequisite_id"),
                dependency.get("dependent_id"),
            }

            invalid_ids = output_ids - stage_1_ids

            if invalid_ids:
                issues.append({
                    "model": model_name,
                    "item_index": item_index,
                    "issue_type": "invented_or_invalid_dependency_id",
                    "allowed_ids": sorted(stage_1_ids),
                    "found_ids": sorted(output_ids),
                    "invalid_ids": sorted(invalid_ids),
                    "dependency_index": dependency_index,
                })

    return issues

In [8]:
all_issues = []

for file_path in RESULT_FILES:
    with file_path.open("r", encoding="utf-8") as file:
        results = json.load(file)

    issues = validate_dependency_ids(results, file_path.name)

    for issue in issues:
        issue["file"] = file_path.name

    all_issues.extend(issues)

issues_df = pd.DataFrame(all_issues)

if issues_df.empty:
    print("All dependency IDs are consistent in every result file.")
else:
    print(f"Found {len(issues_df)} ID consistency issue(s).")
    display(issues_df)

All dependency IDs are consistent in every result file.


## Check ID consistency of the input and the ground truth

In [10]:
def build_user_story_table(results, metadata):
    """
    Build a deduplicated US_ID / US_text table from:
    - pairwise annotated results: kept stories
    - metadata: ill-formed stories
    """
    stories = {}

    # Kept stories from pairwise result entries
    for item in results:
        story_a_id = item["step_1_match"]["story_a_id"]
        story_b_id = item["step_1_match"]["story_b_id"]

        stories[story_a_id] = item["story_a"]
        stories[story_b_id] = item["story_b"]

    # Ill-formed stories exist only in metadata
    for story in metadata.get("ill_formed_stories", []):
        stories[story["id"]] = story["text"]

    return pd.DataFrame(
        [
            {"US_ID": us_id, "US_text": us_text}
            for us_id, us_text in stories.items()
        ]
    )

In [13]:
result_file = PREDICTION_DIR / "llama3.1_8b_annotated_gt_results.json"
metadata_file = PREDICTION_DIR / "llama3.1_8b_annotated_gt_meta.json"

with result_file.open("r", encoding="utf-8") as file:
    results = json.load(file)

with metadata_file.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

user_stories_df = build_user_story_table(results, metadata)

display(user_stories_df)

,US_ID,US_text
0,US-002,"As a returning user, I want to log in my accou..."
1,US-003,"As a user, I want to walk through a short, gui..."
2,US-004,"As a traveler, I want to import a location fro..."
3,US-005,"As a traveler, I want to create named lists (e..."
4,US-006,"As a traveler, I want to attach private notes ..."
5,US-007,"As a traveler, I want to open the original Tik..."
6,US-009,"As a traveler, I want to manually add a spot b..."
7,US-010,"As a traveler, I want to change the name of an..."
8,US-011,"As a traveler, I want to see all my saved spot..."
9,US-012,"As a traveler, I want to filter the map to sho..."


In [15]:
BASELINE_DIR = Path("baseline")
GROUND_TRUTH_FILE = Path(BASELINE_DIR / "Ground_truth_OBD_dependency.xlsx")

ground_truth_df = pd.read_excel(GROUND_TRUTH_FILE, sheet_name="USs")

print(ground_truth_df.columns.tolist())
display(ground_truth_df)

['US_ID', 'US_Text']


,US_ID,US_Text
0,US_1,"As a new user, I want to sign up using my emai..."
1,US_2,"As a returning user, I want to log in my accou..."
2,US_3,"As a user, I want to walk through a short, gui..."
3,US_4,"As a traveler, I want to import a location fro..."
4,US_5,"As a traveler, I want to create named lists (e..."
5,US_6,"As a traveler, I want to attach private notes ..."
6,US_7,"As a traveler, I want to open the original Tik..."
7,US_8,"As a traveler, I want to remove a saved spot f..."
8,US_9,"As a traveler, I want to manually add a spot b..."
9,US_10,"As a traveler, I want to change the name of an..."


In [27]:
def normalize_us_id(value):
    number = int(re.search(r"\d+", str(value)).group())
    return f"US-{number:03d}"


def normalize_text(value):
    return " ".join(str(value).split())

In [28]:
output_table = user_stories_df.copy()
ground_truth_table = ground_truth_df.copy()

output_table["ID"] = output_table["US_ID"].apply(normalize_us_id)
ground_truth_table["ID"] = ground_truth_table["US_ID"].apply(normalize_us_id)

In [29]:
output_texts = dict(zip(output_table["ID"], output_table["US_text"]))
ground_truth_texts = dict(zip(ground_truth_table["ID"], ground_truth_table["US_Text"]))

In [32]:
results = []

all_ids = set(output_texts) | set(ground_truth_texts)

for us_id in sorted(all_ids):
    output_text = output_texts.get(us_id)
    ground_truth_text = ground_truth_texts.get(us_id)

    if output_text is None:
        result = "MISSING_FROM_OUTPUT"

    elif ground_truth_text is None:
        result = "NOT_IN_GROUND_TRUTH"

    elif normalize_text(output_text) == normalize_text(ground_truth_text):
        result = "MATCH"

    else:
        result = "TEXT_MISMATCH"

    results.append({
        "US_ID": us_id,
        "output_text": output_text,
        "ground_truth_text": ground_truth_text,
        "result": result,
    })

In [35]:
print(all_ids)

{'US-002', 'US-007', 'US-001', 'US-006', 'US-011', 'US-005', 'US-015', 'US-010', 'US-008', 'US-020', 'US-016', 'US-003', 'US-012', 'US-009', 'US-018', 'US-019', 'US-013', 'US-014', 'US-004', 'US-017'}


In [33]:
comparison_df = pd.DataFrame(results)

display(comparison_df)

,US_ID,output_text,ground_truth_text,result
0,US-001,"As a new user, I want to sign up using my emai...","As a new user, I want to sign up using my emai...",MATCH
1,US-002,"As a returning user, I want to log in my accou...","As a returning user, I want to log in my accou...",MATCH
2,US-003,"As a user, I want to walk through a short, gui...","As a user, I want to walk through a short, gui...",MATCH
3,US-004,"As a traveler, I want to import a location fro...","As a traveler, I want to import a location fro...",MATCH
4,US-005,"As a traveler, I want to create named lists (e...","As a traveler, I want to create named lists (e...",MATCH
5,US-006,"As a traveler, I want to attach private notes ...","As a traveler, I want to attach private notes ...",MATCH
6,US-007,"As a traveler, I want to open the original Tik...","As a traveler, I want to open the original Tik...",MATCH
7,US-008,"As a traveler, I want to remove a saved spot f...","As a traveler, I want to remove a saved spot f...",MATCH
8,US-009,"As a traveler, I want to manually add a spot b...","As a traveler, I want to manually add a spot b...",MATCH
9,US-010,"As a traveler, I want to change the name of an...","As a traveler, I want to change the name of an...",MATCH
